# IBKR NQ Globex validation v0.2.0
Uses existing NQ quarterly consolidated Parquets. If the Colab Git checkout has uncommitted changes, it is preserved and this runner uses a separate fresh checkout without resetting or deleting anything.

In [ ]:
import importlib, subprocess, sys, tempfile
from pathlib import Path

# Never reset, clean or overwrite an existing Colab checkout.
PREFERRED_REPO = Path("/content/IBKR-market-data-cleaning-pipeline")
REPO = PREFERRED_REPO
URL = "https://github.com/hh4832/IBKR-market-data-cleaning-pipeline.git"


def git_out(*args):
    return subprocess.check_output(
        ["git", "-C", str(PREFERRED_REPO), *args], text=True
    ).strip()


def fresh_isolated_clone():
    # A unique ephemeral Colab checkout, NOT a replacement for the user's
    # dirty directory. Do not put it under the source repo or mounted Drive.
    temporary = Path(tempfile.mkdtemp(prefix="ibkr-nq-qc-clean-", dir="/content"))
    destination = temporary / "repo"
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", URL, str(destination)],
        check=True,
    )
    return destination


if (PREFERRED_REPO / ".git").is_dir():
    origin = git_out("remote", "get-url", "origin")
    if origin.rstrip("/").removesuffix(".git") != URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Unexpected Git remote: {origin}; aborting")
    dirty = git_out("status", "--porcelain")
    if dirty:
        print("Existing checkout has uncommitted changes; preserving it untouched:")
        print("\n".join(dirty.splitlines()[:20]))
        REPO = fresh_isolated_clone()
        print("Using a separate clean clone:", REPO)
    else:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
elif PREFERRED_REPO.exists():
    print("Existing path is not a Git checkout; preserving it untouched.")
    REPO = fresh_isolated_clone()
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)],
        check=True,
    )

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", str(REPO)+"[colab]"], check=True)
src = str(REPO / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
for name in list(sys.modules):
    if name == "ibkr_qc" or name.startswith("ibkr_qc."):
        del sys.modules[name]
importlib.invalidate_caches()
from ibkr_qc.nq_pipeline import NQRunConfig, run_nq_qc
print("NQ QC checkout:", REPO)
print("HEAD:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short=12", "HEAD"], text=True).strip())


## Verified Cache and output folders
The Drive API checks the exact folder hierarchy. The existing Cache is read from MyDrive/00Quant_Research/00_Cache. Reports are staged locally and uploaded to the specified output Drive folder. The notebook will fail rather than silently create a duplicate cache at a wrong path.

In [ ]:
from google.colab import drive
from pathlib import Path
from ibkr_qc.drive_io import authorized_drive

drive.mount("/content/drive")
service = authorized_drive()

# Verified Google Drive hierarchy: MyDrive / 00Quant_Research
QUANT_ROOT_ID = "1kRfLhTLdHevVuFkEzdZH9M5wEui7JuEg"
CACHE_PARENT_ID = "16RheGRQUz-VWhvQJA_2Be2o7BM0zBU1u"
CACHE_FOLDER_ID = "1ExEXrUdnO1mh-MTalmJ8n-fLa362SX91"
SOURCE_FOLDER_ID = "1C0wfYpT8zioRmZxBnqO7d79-zJyD_S5w"
OUTPUT_PARENT_ID = "1Kl7s_3Y02tGeEbLbzF-LhoyEMyCtG129"

root = service.files().get(fileId="root", fields="id,name", supportsAllDrives=True).execute()
folder_chain = (
    (QUANT_ROOT_ID, "00Quant_Research", root["id"]),
    (CACHE_PARENT_ID, "00_Cache", QUANT_ROOT_ID),
    (CACHE_FOLDER_ID, "IBKR_NQ_GLOBEX_QC_CACHE_V020", CACHE_PARENT_ID),
    (OUTPUT_PARENT_ID, "IBKR_NQ_Globex_QC", QUANT_ROOT_ID),
)
for folder_id, expected_name, expected_parent in folder_chain:
    info = service.files().get(
        fileId=folder_id, fields="id,name,mimeType,parents",
        supportsAllDrives=True
    ).execute()
    if (info.get("mimeType") != "application/vnd.google-apps.folder"
        or info.get("name") != expected_name
        or info.get("parents") != [expected_parent]):
        raise RuntimeError(
            f"Google Drive folder moved or mismatched: {expected_name} ({folder_id}). "
            f"Found name={info.get('name')}, parents={info.get('parents')}."
        )

MAX_CONTRACTS = 1  # PILOT; set to 0 to validate ALL nine contracts.
TOLERANCE_MINUTES = 2

# Use the real moved cache location. Never mkdir a duplicate empty cache.
CACHE_DIR = Path("/content/drive/MyDrive/00Quant_Research/00_Cache/IBKR_NQ_GLOBEX_QC_CACHE_V020")
if not CACHE_DIR.is_dir():
    raise RuntimeError(f"Existing NQ cache not found: {CACHE_DIR}. Check Drive mount and folder hierarchy.")
EXPECTED_CACHED_PILOT = CACHE_DIR / "1eO6FR6eNqbH1wrZMVaF4CQlkYmOhxP_c_consolidated.parquet"
if not EXPECTED_CACHED_PILOT.is_file():
    raise RuntimeError("Previously cached NQZ6 Parquet missing. Refusing to silently recreate cache.")

# OUTPUT_DIR is temporary Colab staging; Drive API archives each run
# to a new subfolder under OUTPUT_PARENT_ID, preserving the Pilot.
OUTPUT_DIR = Path("/content/ibkr_nq_qc_outputs")
print("Verified Cache:", CACHE_DIR)
print("Existing Pilot NQZ6 Parquet:", EXPECTED_CACHED_PILOT.name)
print("Archived output folder ID:", OUTPUT_PARENT_ID)
print("Local temporary staging:", OUTPUT_DIR)
print("Run mode:", "FULL" if MAX_CONTRACTS == 0 else "PILOT")


## Run QA
CME special holidays/halts and IBKR bar-label convention remain unverified. Per-contract only: no rollover selection or backadjustment. research_ready is always false in this validation phase.

In [ ]:
from ibkr_qc.nq_pipeline import NQRunConfig, run_nq_qc
import json
import pandas as pd

cfg = NQRunConfig(source_folder_id=SOURCE_FOLDER_ID,
    output_parent_id=OUTPUT_PARENT_ID, max_contracts=MAX_CONTRACTS,
    tolerance_minutes=TOLERANCE_MINUTES, upload=True).validate()
run_dir = run_nq_qc(service, cfg, cache_root=CACHE_DIR, output_root=OUTPUT_DIR)
info = json.loads((run_dir/"run_info.json").read_text(encoding="utf-8"))
print({k:info[k] for k in (
    "mode", "status", "total_contract_folders", "successful_contracts",
    "globex_expected_standard_minutes", "possible_missing_standard_minutes",
    "taipei_candidate_anchor_rows", "research_ready",
)})
display(pd.read_csv(run_dir/"daily_globex_coverage.csv").head(10))
display(pd.read_csv(run_dir/"taipei_candidate_anchor_prices.csv").head(10))
print("Local outputs:",run_dir)